# Stage 3 — DM-Count official-checkpoint preflight

This notebook evaluates the official ShanghaiTech Part A checkpoint on all 182 test images. It verifies the pinned code, data contract, weights, and paper-number gap; it does not replace the remaining three-seed faithful and clean training lanes.

Google Drive is preferred for persistent data and results. If Colab credential propagation fails, the notebook falls back to ephemeral `/content` storage without altering the Drive copy.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
import sys
import time
import urllib.request
import zipfile

import numpy as np
import torch

PIN = 'cc5f2132e0d1328909f31b6d665b8e0b15c30467'
DATASET_URL = 'https://www.kaggle.com/api/v1/datasets/download/tthien/shanghaitech'
CHECKPOINT_ID = '13dBlTb2N2gN5x3wAQ3Bv3sdW_qC9gaMf'
PAPER_MAE = 59.7
PAPER_RMSE = 95.7

print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'

In [ ]:
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OK = Path('/content/drive/MyDrive').is_dir()
except Exception as exc:
    print(f'Drive unavailable; using ephemeral runtime storage: {exc}')

if DRIVE_OK:
    PROJECT_ROOT = Path('/content/drive/MyDrive/DroneAI')
    ARCHIVE = PROJECT_ROOT / 'datasets' / 'shanghaitech' / 'shanghaitech.zip'
    DATA_ROOT = PROJECT_ROOT / 'datasets' / 'shanghaitech' / 'extracted'
    CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'dm-count' / 'official' / 'model_sh_A.pth'
    RESULT_DIR = PROJECT_ROOT / 'runs' / 'stage-3' / 'official-checkpoint'
else:
    PROJECT_ROOT = Path('/content/droneai-stage3')
    ARCHIVE = PROJECT_ROOT / 'shanghaitech.zip'
    DATA_ROOT = PROJECT_ROOT / 'data'
    CHECKPOINT = PROJECT_ROOT / 'model_sh_A.pth'
    RESULT_DIR = PROJECT_ROOT / 'results'

for path in (ARCHIVE.parent, DATA_ROOT, CHECKPOINT.parent, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print({'drive_ok': DRIVE_OK, 'project_root': str(PROJECT_ROOT)})

In [ ]:
PART_A = DATA_ROOT / 'ShanghaiTech' / 'part_A'
UPSTREAM_DIR = Path('/content/DM-Count')

if not ARCHIVE.exists():
    print('Downloading ShanghaiTech archive...')
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE)
if not PART_A.is_dir():
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(DATA_ROOT)

if not CHECKPOINT.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download(id=CHECKPOINT_ID, output=str(CHECKPOINT), quiet=False)

if not (UPSTREAM_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/cvlab-stonybrook/DM-Count.git', str(UPSTREAM_DIR)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'checkout', PIN], check=True)
actual_pin = subprocess.check_output(['git', '-C', str(UPSTREAM_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_pin == PIN, actual_pin

train_images = list((PART_A / 'train_data' / 'images').glob('*.jpg'))
test_images = list((PART_A / 'test_data' / 'images').glob('*.jpg'))
train_gt = list((PART_A / 'train_data' / 'ground-truth').glob('*.mat'))
test_gt = list((PART_A / 'test_data' / 'ground-truth').glob('*.mat'))
counts = {'train_images': len(train_images), 'train_gt': len(train_gt), 'test_images': len(test_images), 'test_gt': len(test_gt)}
assert counts == {'train_images': 300, 'train_gt': 300, 'test_images': 182, 'test_gt': 182}, counts
print({'commit': actual_pin, 'counts': counts, 'checkpoint_bytes': CHECKPOINT.stat().st_size})

In [ ]:
sys.path.insert(0, str(UPSTREAM_DIR))
from datasets.crowd import Crowd_sh
from models import vgg19
from torch.utils.data import DataLoader

device = torch.device('cuda')
dataset = Crowd_sh(str(PART_A / 'test_data'), 512, 8, method='val')
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=1, pin_memory=True)
model = vgg19().to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
model.eval()

rows, errors = [], []
started = time.time()
with torch.no_grad():
    for index, (inputs, count, name) in enumerate(dataloader, 1):
        outputs, _ = model(inputs.to(device, non_blocking=True))
        ground_truth = float(count[0].item())
        prediction = float(outputs.sum().item())
        error = ground_truth - prediction
        sample = name[0] if isinstance(name, (list, tuple)) else str(name)
        rows.append({'index': index, 'name': str(sample), 'ground_truth': ground_truth, 'prediction': prediction, 'error_gt_minus_pred': error, 'absolute_error': abs(error)})
        errors.append(error)
        if index == 1 or index % 20 == 0 or index == len(dataset):
            print(f'[{index}/{len(dataset)}] {sample}: gt={ground_truth:.1f}, pred={prediction:.1f}, abs_err={abs(error):.1f}')

errors = np.asarray(errors, dtype=np.float64)
mae = float(np.mean(np.abs(errors)))
rmse = float(np.sqrt(np.mean(np.square(errors))))
elapsed = time.time() - started
result = {
    'stage': 3, 'status': 'PREFLIGHT_PASS_STAGE_IN_PROGRESS', 'kind': 'official_checkpoint_preflight',
    'dataset': 'ShanghaiTech Part A', 'test_samples': len(rows), 'upstream_commit': PIN, 'checkpoint': CHECKPOINT.name,
    'evaluation_protocol': {'crop_size': 512, 'downsample_ratio': 8, 'batch_size': 1, 'method': 'val', 'prediction': 'sum of the predicted density map'},
    'environment': {'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0)},
    'metrics': {'mae': mae, 'rmse': rmse}, 'paper_targets': {'mae': PAPER_MAE, 'rmse': PAPER_RMSE},
    'relative_gap_pct': {'mae': (mae / PAPER_MAE - 1) * 100, 'rmse': (rmse / PAPER_RMSE - 1) * 100},
    'within_5pct': {'mae': abs(mae / PAPER_MAE - 1) <= 0.05, 'rmse': abs(rmse / PAPER_RMSE - 1) <= 0.05},
    'elapsed_seconds': elapsed
}
assert result['within_5pct'] == {'mae': True, 'rmse': True}, result

(RESULT_DIR / 'official_checkpoint_metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
with (RESULT_DIR / 'official_checkpoint_predictions.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(json.dumps(result, indent=2))
print({'result_dir': str(RESULT_DIR)})